# MTA Demo Analysis Notebook

Compact Databricks-ready notebook for slide-level data overview, selection-bias diagnostics, and optional SHAP vs doubly robust comparison.

Assumptions:
- `mta_table` is a PySpark DataFrame or a registered Spark table.
- One row per customer, with binary channel interaction columns.
- Aggregated outputs are small and safe to convert to pandas for plotting/export.


In [ ]:
# ===== 1) CONFIG: update these names if needed =====

# Source table/dataframe. If a Python variable with this name exists, the notebook uses it.
# Otherwise it tries spark.table(TABLE_NAME).
TABLE_NAME = "mta_table"

CUSTOMER_ID_COL = "cost_ID"
CONVERTED_COL = "Converted"
PROPENSITY_COL = "SC_4875001"      # baseline any-credit-card propensity
NUM_TOUCHPOINTS_COL = "num_touchpoints"
CHANNEL_PREFIX = "channel_"

# Optional attribution table. Set to None if unavailable.
# Expected columns: interaction_name, shap_value, dr_value
ATTRIBUTION_TABLE_NAME = None       # e.g., "mta_attr_results" or None
ATTR_INTERACTION_COL = "interaction_name"
ATTR_SHAP_COL = "shap_value"
ATTR_DR_COL = "dr_value"

# Optional manual mapping if automatic channel parsing is wrong.
# Example: {"channel_scio_click": "scio", "channel_mobile_app_interested": "mobile"}
CHANNEL_NAME_OVERRIDES = {}

# Plot/export options
TOP_N = 15
TOP_CHANNELS_FOR_DECOLE_PLOT = 5
OUTPUT_DIR = "/dbfs/FileStore/mta_demo_outputs"  # change or set to None to skip saving files


In [ ]:
# ===== 2) Imports and load dataframe =====
from pyspark.sql import functions as F
from pyspark.sql import Window
from functools import reduce
import pandas as pd
import numpy as np
import os

try:
    dbutils  # Databricks only
except NameError:
    dbutils = None

if TABLE_NAME in globals():
    df = globals()[TABLE_NAME]
else:
    df = spark.table(TABLE_NAME)

if OUTPUT_DIR:
    os.makedirs(OUTPUT_DIR, exist_ok=True)

# Normalize conversion to numeric 0/1 for aggregation
working_df = df.withColumn(CONVERTED_COL, F.col(CONVERTED_COL).cast("double"))

channel_interaction_cols = [
    c for c in working_df.columns
    if c.startswith(CHANNEL_PREFIX) and c not in {CONVERTED_COL, CUSTOMER_ID_COL, PROPENSITY_COL, NUM_TOUCHPOINTS_COL}
]

print(f"Found {len(channel_interaction_cols)} channel interaction columns")
print(channel_interaction_cols[:20])


In [ ]:
# ===== 3) Helper functions =====

def infer_channel_name(col_name: str) -> str:
    if col_name in CHANNEL_NAME_OVERRIDES:
        return CHANNEL_NAME_OVERRIDES[col_name]
    base = col_name.replace(CHANNEL_PREFIX, "", 1)
    # Default: channel is the first token before interaction type.
    # Adjust CHANNEL_NAME_OVERRIDES if your naming convention is different.
    return base.split("_")[0]


def exposure_summary(input_df, feature_cols, feature_name_col="feature_name"):
    """Compute exposure, conversion, lift, and propensity bias metrics for binary feature columns."""
    total_customers = input_df.select(F.countDistinct(CUSTOMER_ID_COL).alias("n")).first()["n"]
    total_converters = input_df.agg(F.sum(F.col(CONVERTED_COL)).alias("y")).first()["y"]
    overall_cvr = float(total_converters) / total_customers if total_customers else None

    agg_exprs = []
    for c in feature_cols:
        t = F.coalesce(F.col(c).cast("int"), F.lit(0))
        agg_exprs += [
            F.sum(t).alias(f"{c}__exposed"),
            F.sum(F.when(t == 1, F.col(CONVERTED_COL)).otherwise(0)).alias(f"{c}__conv_exp"),
            F.sum(F.when(t == 0, F.col(CONVERTED_COL)).otherwise(0)).alias(f"{c}__conv_unexp"),
            F.sum(F.when(t == 0, 1).otherwise(0)).alias(f"{c}__unexposed"),
            F.avg(F.when(t == 1, F.col(PROPENSITY_COL))).alias(f"{c}__avg_prop_exp"),
            F.avg(F.when(t == 0, F.col(PROPENSITY_COL))).alias(f"{c}__avg_prop_unexp"),
        ]

    row = input_df.agg(*agg_exprs).first().asDict()
    records = []
    for c in feature_cols:
        exposed = row.get(f"{c}__exposed") or 0
        unexposed = row.get(f"{c}__unexposed") or 0
        conv_exp = row.get(f"{c}__conv_exp") or 0
        conv_unexp = row.get(f"{c}__conv_unexp") or 0
        cvr_exp = float(conv_exp) / exposed if exposed else None
        cvr_unexp = float(conv_unexp) / unexposed if unexposed else None
        raw_lift = cvr_exp - cvr_unexp if cvr_exp is not None and cvr_unexp is not None else None
        records.append({
            feature_name_col: c,
            "exposed_customers": int(exposed),
            "exposure_rate": float(exposed) / total_customers if total_customers else None,
            "converters_exposed": int(conv_exp),
            "conversion_rate_exposed": cvr_exp,
            "converters_unexposed": int(conv_unexp),
            "conversion_rate_unexposed": cvr_unexp,
            "raw_lift": raw_lift,
            "avg_propensity_exposed": row.get(f"{c}__avg_prop_exp"),
            "avg_propensity_unexposed": row.get(f"{c}__avg_prop_unexp"),
            "avg_propensity_gap": (row.get(f"{c}__avg_prop_exp") or 0) - (row.get(f"{c}__avg_prop_unexp") or 0),
            "overall_conversion_rate": overall_cvr,
        })

    pdf = pd.DataFrame(records).sort_values("exposed_customers", ascending=False)
    return spark.createDataFrame(pdf), pdf


def save_pdf(pdf, name):
    if OUTPUT_DIR:
        path = f"{OUTPUT_DIR}/{name}.csv"
        pdf.to_csv(path, index=False)
        print(f"Saved: {path}")


## Slide 1: Overall data overview

Use this for a compact population/conversion/touchpoint summary.

In [ ]:
# ===== 4) Overall summary =====
base_metrics = working_df.agg(
    F.countDistinct(CUSTOMER_ID_COL).alias("total_customers"),
    F.count("*").alias("rows"),
    F.sum(F.col(CONVERTED_COL)).alias("converters"),
    F.avg(F.col(CONVERTED_COL)).alias("overall_conversion_rate"),
    F.sum(F.col(NUM_TOUCHPOINTS_COL)).alias("total_touchpoints"),
    F.avg(F.col(NUM_TOUCHPOINTS_COL)).alias("avg_touchpoints"),
    F.avg(F.col(PROPENSITY_COL)).alias("avg_baseline_propensity"),
).toPandas()

# Approx quantiles for touchpoints
q = working_df.approxQuantile(NUM_TOUCHPOINTS_COL, [0.25, 0.5, 0.75, 0.9], 0.01)
base_metrics["touchpoints_p25"] = q[0] if len(q) > 0 else None
base_metrics["touchpoints_median"] = q[1] if len(q) > 1 else None
base_metrics["touchpoints_p75"] = q[2] if len(q) > 2 else None
base_metrics["touchpoints_p90"] = q[3] if len(q) > 3 else None

summary_sdf = spark.createDataFrame(base_metrics)
display(summary_sdf)
save_pdf(base_metrics, "01_overall_summary")

# Baseline propensity by conversion label
prop_by_y = working_df.groupBy(CONVERTED_COL).agg(
    F.countDistinct(CUSTOMER_ID_COL).alias("customers"),
    F.avg(PROPENSITY_COL).alias("avg_baseline_propensity"),
    F.expr(f"percentile_approx({PROPENSITY_COL}, 0.5)").alias("median_baseline_propensity"),
).orderBy(CONVERTED_COL)

display(prop_by_y)
save_pdf(prop_by_y.toPandas(), "02_propensity_by_conversion")


## Slide 1/2: Channel and interaction reach

This creates two slide-ready tables:
- interaction-level exposure/conversion/lift
- channel-level exposure/conversion/lift


In [ ]:
# ===== 5) Interaction-level exposure/conversion/lift =====
interaction_sdf, interaction_pdf = exposure_summary(
    working_df,
    channel_interaction_cols,
    feature_name_col="interaction_name"
)

display(interaction_sdf.orderBy(F.desc("exposed_customers")))
save_pdf(interaction_pdf, "03_interaction_exposure_summary")


In [ ]:
# ===== 6) Channel-level summary =====
channel_to_cols = {}
for c in channel_interaction_cols:
    ch = infer_channel_name(c)
    channel_to_cols.setdefault(ch, []).append(c)

channel_df = working_df
channel_present_cols = []
for ch, cols in channel_to_cols.items():
    present_col = f"channel_present__{ch}"
    expr = reduce(lambda a, b: a | b, [F.coalesce(F.col(c).cast("int"), F.lit(0)) == 1 for c in cols])
    channel_df = channel_df.withColumn(present_col, F.when(expr, 1).otherwise(0))
    channel_present_cols.append(present_col)

channel_sdf, channel_pdf = exposure_summary(
    channel_df,
    channel_present_cols,
    feature_name_col="channel_name"
)
channel_pdf["channel_name"] = channel_pdf["channel_name"].str.replace("channel_present__", "", regex=False)
channel_sdf = spark.createDataFrame(channel_pdf)

display(channel_sdf.orderBy(F.desc("exposed_customers")))
save_pdf(channel_pdf, "04_channel_exposure_summary")


## Slide 2: Touchpoint depth

This shows whether customers with more touches have higher observed conversion rates.

In [ ]:
# ===== 7) Touchpoint-depth table and plot =====
touch_df = working_df.withColumn(
    "touchpoint_bucket",
    F.when(F.col(NUM_TOUCHPOINTS_COL).isNull(), "missing")
     .when(F.col(NUM_TOUCHPOINTS_COL) == 0, "0")
     .when(F.col(NUM_TOUCHPOINTS_COL) == 1, "1")
     .when(F.col(NUM_TOUCHPOINTS_COL) == 2, "2")
     .when(F.col(NUM_TOUCHPOINTS_COL) == 3, "3")
     .when(F.col(NUM_TOUCHPOINTS_COL) == 4, "4")
     .otherwise("5+")
)

total_customers = working_df.select(F.countDistinct(CUSTOMER_ID_COL).alias("n")).first()["n"]

touch_sdf = touch_df.groupBy("touchpoint_bucket").agg(
    F.countDistinct(CUSTOMER_ID_COL).alias("customers"),
    F.sum(CONVERTED_COL).alias("converters"),
    F.avg(CONVERTED_COL).alias("conversion_rate"),
).withColumn("share_of_customers", F.col("customers") / F.lit(total_customers))

bucket_order = {"0": 0, "1": 1, "2": 2, "3": 3, "4": 4, "5+": 5, "missing": 6}
touch_pdf = touch_sdf.toPandas()
touch_pdf["bucket_order"] = touch_pdf["touchpoint_bucket"].map(bucket_order)
touch_pdf = touch_pdf.sort_values("bucket_order").drop(columns=["bucket_order"])

display(spark.createDataFrame(touch_pdf))
save_pdf(touch_pdf, "05_touchpoint_depth")

# Plot: conversion rate by touchpoint bucket
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.bar(touch_pdf["touchpoint_bucket"], touch_pdf["conversion_rate"])
plt.title("Observed conversion rate by number of touchpoints")
plt.xlabel("Number of touchpoints")
plt.ylabel("Conversion rate")
plt.tight_layout()
if OUTPUT_DIR:
    plt.savefig(f"{OUTPUT_DIR}/plot_touchpoint_conversion_rate.png", dpi=200, bbox_inches="tight")
plt.show()


## Slide 3: Selection-bias diagnostic

This is the core story: some channels may be concentrated among customers who already had high baseline credit-card propensity.

In [ ]:
# ===== 8) Baseline propensity deciles =====
# Use ntile for equal-sized deciles. This requires a sort but is fine for a demo-sized run.
w = Window.orderBy(F.col(PROPENSITY_COL).asc_nulls_first())
decile_df = channel_df.withColumn("propensity_decile", F.ntile(10).over(w))

decile_sdf = decile_df.groupBy("propensity_decile").agg(
    F.countDistinct(CUSTOMER_ID_COL).alias("customers"),
    F.sum(CONVERTED_COL).alias("converters"),
    F.avg(CONVERTED_COL).alias("conversion_rate"),
    F.avg(PROPENSITY_COL).alias("avg_baseline_propensity"),
).orderBy("propensity_decile")

display(decile_sdf)
decile_pdf = decile_sdf.toPandas()
save_pdf(decile_pdf, "06_propensity_decile_summary")

# Plot: conversion rate by baseline propensity decile
plt.figure(figsize=(8, 4))
plt.plot(decile_pdf["propensity_decile"], decile_pdf["conversion_rate"], marker="o")
plt.title("Observed conversion rate by baseline propensity decile")
plt.xlabel("Baseline any-card propensity decile")
plt.ylabel("Conversion rate")
plt.xticks(range(1, 11))
plt.tight_layout()
if OUTPUT_DIR:
    plt.savefig(f"{OUTPUT_DIR}/plot_conversion_by_propensity_decile.png", dpi=200, bbox_inches="tight")
plt.show()


In [ ]:
# ===== 9) Channel exposure by propensity decile =====
# Select top channels by reach to avoid overcrowding the plot.
top_channels = (
    channel_pdf.sort_values("exposed_customers", ascending=False)
    .head(TOP_CHANNELS_FOR_DECOLE_PLOT)["channel_name"]
    .tolist()
)
top_channel_cols = [f"channel_present__{ch}" for ch in top_channels]

exprs = [F.avg(F.col(c).cast("double")).alias(c.replace("channel_present__", "")) for c in top_channel_cols]
channel_decile_sdf = decile_df.groupBy("propensity_decile").agg(*exprs).orderBy("propensity_decile")
channel_decile_pdf = channel_decile_sdf.toPandas()

display(channel_decile_sdf)
save_pdf(channel_decile_pdf, "07_channel_exposure_by_propensity_decile")

# Plot: exposure rate by decile for top channels
plt.figure(figsize=(9, 5))
for ch in top_channels:
    plt.plot(channel_decile_pdf["propensity_decile"], channel_decile_pdf[ch], marker="o", label=ch)
plt.title("Channel exposure rate by baseline propensity decile")
plt.xlabel("Baseline any-card propensity decile")
plt.ylabel("Exposure rate")
plt.xticks(range(1, 11))
plt.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
if OUTPUT_DIR:
    plt.savefig(f"{OUTPUT_DIR}/plot_channel_exposure_by_propensity_decile.png", dpi=200, bbox_inches="tight")
plt.show()


## Slide 4: Optional SHAP vs doubly robust comparison

Run this only if your attribution table is available. It joins SHAP/DR results with interaction-level exposure/conversion metrics.

In [ ]:
# ===== 10) Optional: load attribution results and join with interaction summary =====
if ATTRIBUTION_TABLE_NAME is None:
    print("ATTRIBUTION_TABLE_NAME is None. Set it in cell 1 if you want SHAP vs DR comparison.")
    attr_join_pdf = None
else:
    if ATTRIBUTION_TABLE_NAME in globals():
        attr_df = globals()[ATTRIBUTION_TABLE_NAME]
    else:
        attr_df = spark.table(ATTRIBUTION_TABLE_NAME)

    attr_clean = attr_df.select(
        F.col(ATTR_INTERACTION_COL).alias("interaction_name"),
        F.col(ATTR_SHAP_COL).cast("double").alias("shap_value"),
        F.col(ATTR_DR_COL).cast("double").alias("dr_value"),
    )

    attr_join_sdf = attr_clean.join(interaction_sdf, on="interaction_name", how="left") \
        .withColumn("dr_minus_shap", F.col("dr_value") - F.col("shap_value")) \
        .withColumn(
            "percent_change_from_shap_to_dr",
            F.when(F.abs(F.col("shap_value")) > 1e-12, (F.col("dr_value") - F.col("shap_value")) / F.abs(F.col("shap_value")))
        )

    display(attr_join_sdf.orderBy(F.desc(F.abs(F.col("shap_value")))))
    attr_join_pdf = attr_join_sdf.toPandas()
    save_pdf(attr_join_pdf, "08_shap_dr_joined_summary")


In [ ]:
# ===== 11) Optional: SHAP vs DR plots =====
if attr_join_pdf is None:
    print("No attribution data loaded. Skipping plots.")
else:
    plot_pdf = attr_join_pdf.dropna(subset=["shap_value", "dr_value"]).copy()
    plot_pdf["abs_shap"] = plot_pdf["shap_value"].abs()
    top_pdf = plot_pdf.sort_values("abs_shap", ascending=False).head(TOP_N).sort_values("abs_shap")

    # 1) Side-by-side SHAP vs DR
    x = np.arange(len(top_pdf))
    width = 0.4
    plt.figure(figsize=(10, 6))
    plt.barh(x - width/2, top_pdf["shap_value"], height=width, label="TreeSHAP")
    plt.barh(x + width/2, top_pdf["dr_value"], height=width, label="Doubly robust")
    plt.yticks(x, top_pdf["interaction_name"])
    plt.title("TreeSHAP vs doubly robust attribution")
    plt.xlabel("Attribution value")
    plt.legend()
    plt.tight_layout()
    if OUTPUT_DIR:
        plt.savefig(f"{OUTPUT_DIR}/plot_shap_vs_dr_top_interactions.png", dpi=200, bbox_inches="tight")
    plt.show()

    # 2) Scatter SHAP vs DR
    plt.figure(figsize=(6, 6))
    plt.scatter(plot_pdf["shap_value"], plot_pdf["dr_value"])
    low = min(plot_pdf["shap_value"].min(), plot_pdf["dr_value"].min())
    high = max(plot_pdf["shap_value"].max(), plot_pdf["dr_value"].max())
    plt.plot([low, high], [low, high], linestyle="--")
    plt.title("SHAP vs doubly robust attribution")
    plt.xlabel("TreeSHAP value")
    plt.ylabel("Doubly robust value")
    plt.tight_layout()
    if OUTPUT_DIR:
        plt.savefig(f"{OUTPUT_DIR}/plot_shap_dr_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()

    # 3) Biggest decreases after DR
    drop_pdf = plot_pdf.sort_values("dr_minus_shap").head(TOP_N).sort_values("dr_minus_shap", ascending=False)
    plt.figure(figsize=(10, 6))
    plt.barh(drop_pdf["interaction_name"], drop_pdf["dr_minus_shap"])
    plt.title("Largest decreases from TreeSHAP to doubly robust attribution")
    plt.xlabel("DR - SHAP")
    plt.tight_layout()
    if OUTPUT_DIR:
        plt.savefig(f"{OUTPUT_DIR}/plot_largest_dr_decreases.png", dpi=200, bbox_inches="tight")
    plt.show()

    # 4) Raw lift vs DR
    if "raw_lift" in plot_pdf.columns:
        plt.figure(figsize=(6, 5))
        plt.scatter(plot_pdf["raw_lift"], plot_pdf["dr_value"])
        plt.title("Raw observed lift vs doubly robust attribution")
        plt.xlabel("Raw lift: exposed CVR - unexposed CVR")
        plt.ylabel("Doubly robust value")
        plt.tight_layout()
        if OUTPUT_DIR:
            plt.savefig(f"{OUTPUT_DIR}/plot_raw_lift_vs_dr.png", dpi=200, bbox_inches="tight")
        plt.show()


## Suggested slide usage

- **Data overview slide:** use `01_overall_summary`, `04_channel_exposure_summary`, and the touchpoint conversion plot.
- **Selection-bias slide:** use channel exposure by propensity decile + conversion by propensity decile.
- **Attribution slide:** use SHAP vs DR side-by-side and/or biggest decreases from SHAP to DR.

Recommended wording:
> TreeSHAP explains what the response model learned from observed journeys. Because channel exposure is observational and may be targeted toward already high-propensity customers, we use doubly robust adjustment to reduce observed selection bias.
